<img src='../images/xebia-logo.png' width='300px' align='right' style="padding: 15px">

# Day 3 Morning: Review Session

This notebook brings together everything from the past two days — packaging, code quality, testing, logging, and OOP — into a single hands-on session.

You'll be building a small but complete **Polars-based data pipeline** for the animal shelter dataset, applying all the patterns you've practised.

**Don't worry about Polars syntax** — every cell that needs it has the Polars code already written. Your job is to apply the Python patterns around it.

---

### Topics covered

| # | Topic | From day |
|---|---|---|
| 1 | Packaging & imports | Day 1 — notebook 01 |
| 2 | Code quality & refactoring | Day 1 — notebook 02 |
| 3 | Unit tests & fixtures | Day 1 — notebooks 04, 05 |
| 4 | Logging | Day 2 — notebook 06 |
| 5 | Context managers | Day 2 — OOP track |
| 6 | Classes, inheritance & dunders | Day 2 — OOP track |
| 7 | Putting it all together | Today |

---

### 🟦 Trainer note
> Each section has a **Trainer note** box like this one. These give you talking points, expected answers, and timing suggestions. They are formatted as blockquotes so they stand out clearly and you can collapse/skip them when sharing your screen with participants.

## Setup

Run this cell first. It installs any missing packages and imports everything needed for the session.

In [ ]:
# Run once to make sure all packages are available
import subprocess, sys
packages = ["polars", "pytest", "pytest-ipynb2", "hypothesis"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

In [ ]:
import logging
import re
import time
from contextlib import contextmanager
from functools import wraps
from pathlib import Path

import polars as pl

print("All imports OK  ✓")
print(f"Polars version: {pl.__version__}")

---

## Part 1 — Polars orientation (10 min)

> ### 🟦 Trainer note
> **Timing:** ~10 minutes, trainer-led. No exercise here — just run the cells together and talk through them.
> 
> **Goal:** Give participants enough Polars to not feel lost. Key messages:
> - Polars DataFrames are *immutable* — every operation returns a new DataFrame, no `df['col'] = ...`
> - `pl.col("name")` is how you refer to a column inside an expression
> - `.with_columns()` adds/replaces columns; `.filter()` filters rows; `.select()` picks columns
> - The rest is just Python — all the patterns they've practised work exactly the same

Here is a quick tour of the Polars operations you'll see today. Read through and run each cell.

In [ ]:
# Creating a DataFrame — like pd.DataFrame(...)
df = pl.DataFrame({
    "name":   ["Luna", "Biscuit", "Rex", "Shadow"],
    "type":   ["Cat",  "Dog",    "Dog",  "Cat"],
    "age_yr": [2,      5,        1,      3],
})
df

In [ ]:
# .filter() — like df[df['col'] == value]
df.filter(pl.col("type") == "Dog")

In [ ]:
# .with_columns() — add or replace a column
# Polars DataFrames are immutable: this returns a NEW DataFrame, it does not change df
df.with_columns(
    (pl.col("age_yr") * 365).alias("age_days")
)

In [ ]:
# .select() — choose columns (like df[['col1', 'col2']])
df.select(["name", "type"])

In [ ]:
# Chaining — this is the idiomatic Polars style
result = (
    df
    .filter(pl.col("age_yr") >= 2)
    .with_columns((pl.col("age_yr") * 365).alias("age_days"))
    .select(["name", "type", "age_days"])
)
result

In [ ]:
# .pipe() — pass a DataFrame through a function, exactly like pandas
def add_senior_flag(df: pl.DataFrame) -> pl.DataFrame:
    """Mark animals older than 7 years as senior."""
    return df.with_columns(
        (pl.col("age_yr") > 7).cast(pl.Int8).alias("is_senior")
    )

df.pipe(add_senior_flag)

That's all the Polars you need today. Everything else is Python.

---

## Part 2 — Refactoring (25 min)

> ### 🟦 Trainer note
> **Timing:** ~25 minutes. Participants work in pairs or individually.
> 
> **Recap prompt (2 min):** *"Yesterday we talked about the single responsibility principle — one function should do one thing. What was the problem with the original `add_features()` function?"* (Expected: it did everything in one place, was hard to test, had side effects.)
> 
> **What to watch for:**
> - Each helper function should take a `pl.Series` or `pl.DataFrame` and return something — no mutation
> - The naming convention: public functions vs `_private` helpers
> - The Polars code in each skeleton is complete — participants just need to give it a function body and the right name/signature
> 
> **Model answer key:**
> - `check_has_name`: `return name.str.to_lowercase() != "unknown"`
> - `get_sex`: use `pl.when(...).then(...).otherwise(...)`
> - `compute_age_days`: provided in full — just move into function body
> - `add_features`: call each helper and chain `.with_columns()`

Below is a working but messy data pipeline written in Polars. It does the same thing as the original `add_features()` from Day 1 — but it's all in one place and it mutates nothing (Polars DataFrames are immutable, so this is free!).

Your task: **refactor it** into small, well-named functions.

In [ ]:
# ── The messy version ─────────────────────────────────────────────────────────
# Read this, understand what it does, then refactor it below.

def add_all_features_messy(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_columns(
            # is_dog
            (pl.col("animal_type").str.to_lowercase() == "dog")
            .cast(pl.Int8).alias("is_dog")
        )
        .with_columns(
            # has_name
            (pl.col("name").str.to_lowercase() != "unknown")
            .alias("has_name")
        )
        .with_columns(
            # sex
            pl.when(pl.col("sex_upon_outcome").str.ends_with("Female")).then(pl.lit("female"))
            .when(pl.col("sex_upon_outcome").str.ends_with("Male")).then(pl.lit("male"))
            .otherwise(pl.lit("unknown"))
            .alias("sex")
        )
        .with_columns(
            # neutered
            pl.when(pl.col("sex_upon_outcome").str.to_lowercase().str.contains("neutered|spayed"))
            .then(pl.lit("fixed"))
            .when(pl.col("sex_upon_outcome").str.to_lowercase().str.contains("intact"))
            .then(pl.lit("intact"))
            .otherwise(pl.lit("unknown"))
            .alias("neutered")
        )
        .with_columns(
            # age_days
            pl.when(pl.col("age_upon_outcome").str.contains("year"))
            .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 365)
            .when(pl.col("age_upon_outcome").str.contains("month"))
            .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 30)
            .when(pl.col("age_upon_outcome").str.contains("week"))
            .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 7)
            .otherwise(pl.lit(1))
            .alias("age_days")
        )
    )

### <mark>Exercise 2a: Split into helper functions</mark>

Fill in each function below. The Polars expression for each feature is already written — you just need to put it in the right place.

Each function should:
- Accept a `pl.Series` (a single column) and return a `pl.Series`
- Do exactly one thing
- Have a leading underscore if it's only used inside `add_features()`

In [ ]:
def _check_is_dog(animal_type: pl.Series) -> pl.Series:
    """Return 1 if dog, 0 otherwise."""
    return (animal_type.str.to_lowercase() == "dog").cast(pl.Int8)


def _check_has_name(name: pl.Series) -> pl.Series:
    """Return True if name is not 'unknown'."""
    return name.str.to_lowercase() != "unknown"


def _get_sex(sex_upon_outcome: pl.Series) -> pl.Series:
    """Return 'male', 'female', or 'unknown'."""
    return (
        pl.when(sex_upon_outcome.str.ends_with("Female")).then(pl.lit("female"))
        .when(sex_upon_outcome.str.ends_with("Male")).then(pl.lit("male"))
        .otherwise(pl.lit("unknown"))
    )


def _get_neutered(sex_upon_outcome: pl.Series) -> pl.Series:
    """Return 'fixed', 'intact', or 'unknown'."""
    lowered = sex_upon_outcome.str.to_lowercase()
    return (
        pl.when(lowered.str.contains("neutered|spayed")).then(pl.lit("fixed"))
        .when(lowered.str.contains("intact")).then(pl.lit("intact"))
        .otherwise(pl.lit("unknown"))
    )


def _compute_age_days(age_upon_outcome: pl.Series) -> pl.Series:
    """Convert age string like '2 years' or '3 months' to number of days."""
    return (
        pl.when(age_upon_outcome.str.contains("year"))
        .then(age_upon_outcome.str.extract(r"(\d+)").cast(pl.Int32) * 365)
        .when(age_upon_outcome.str.contains("month"))
        .then(age_upon_outcome.str.extract(r"(\d+)").cast(pl.Int32) * 30)
        .when(age_upon_outcome.str.contains("week"))
        .then(age_upon_outcome.str.extract(r"(\d+)").cast(pl.Int32) * 7)
        .otherwise(pl.lit(1))
    )


def add_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add all engineered features to the DataFrame.

    This is the only public function — callers use this, not the helpers.
    """
    return (
        df
        .with_columns(_check_is_dog(df["animal_type"]).alias("is_dog"))
        .with_columns(_check_has_name(df["name"]).alias("has_name"))
        .with_columns(_get_sex(df["sex_upon_outcome"]).alias("sex"))
        .with_columns(_get_neutered(df["sex_upon_outcome"]).alias("neutered"))
        .with_columns(_compute_age_days(df["age_upon_outcome"]).alias("age_days"))
    )


# ── Test your refactored functions ────────────────────────────────────────────
sample = pl.DataFrame({
    "animal_type":      ["Dog",           "Cat",           "Dog"],
    "name":             ["Biscuit",        "unknown",       "Rex"],
    "sex_upon_outcome": ["Neutered Male",  "Intact Female", "Spayed Female"],
    "age_upon_outcome": ["2 years",        "6 months",      "3 weeks"],
    "outcome_type":     ["Adoption",       "Transfer",      "Return to Owner"],
})

result = add_features(sample)
result


In [ ]:
# ── Test your refactored functions with this small DataFrame ──────────────────
sample = pl.DataFrame({
    "animal_type":      ["Dog",           "Cat",           "Dog"],
    "name":             ["Biscuit",        "unknown",       "Rex"],
    "sex_upon_outcome": ["Neutered Male",  "Intact Female", "Spayed Female"],
    "age_upon_outcome": ["2 years",        "6 months",      "3 weeks"],
    "outcome_type":     ["Adoption",       "Transfer",      "Return to Owner"],
})

result = add_features(sample)
result

**Expected output — check yours matches:**

| name | is_dog | has_name | sex | neutered | age_days |
|------|--------|----------|-----|----------|----------|
| Biscuit | 1 | True | male | fixed | 730 |
| unknown | 0 | False | female | intact | 180 |
| Rex | 1 | True | female | fixed | 21 |

---

## Part 3 — Unit tests & fixtures (30 min)

> ### 🟦 Trainer note
> **Timing:** ~30 minutes. Participants work individually.
> 
> **Recap prompt (2 min):** *"What are the three things a good unit test checks? What is a fixture for?"*
> Expected: (1) the happy path, (2) edge cases, (3) that it raises the right errors. Fixture = shared, reusable setup that keeps tests DRY.
> 
> **What to watch for:**
> - Tests should be small and test one thing each
> - The fixture is already written — they just need to use it
> - For Polars, `assert result.equals(expected)` is the equivalent of `assert_series_equal`
> - `pl.Series([True, False, True])` for boolean series; `.cast(pl.Int8)` for 0/1
> 
> **Common mistake:** Participants try to assert `result == expected` on a Series — remind them Series comparison returns a Series of booleans, not a single True/False. Use `.equals()` instead.
> 
> **Polars testing helpers (put on whiteboard/screen):**
> ```python
> # Check two Series are equal
> assert result.equals(expected)
> 
> # Check a specific value at position i
> assert result[0] == 1
> 
> # Check all values
> assert result.to_list() == [1, 0, 1]
> ```

### <mark>Exercise 3a: Write unit tests for your helper functions</mark>

The fixture `sex_series` is provided. Write at least one test per helper function.

Remember:
- Test the **happy path** (normal input)
- Test at least one **edge case** (e.g. what happens with `"Unknown"` input?)
- Use `assert result.to_list() == [...]` or `assert result.equals(expected)` for Polars Series

In [ ]:
import pytest

@pytest.fixture
def sex_series():
    return pl.Series(["Neutered Male", "Spayed Female", "Intact Male", "Unknown"])

@pytest.fixture
def age_series():
    return pl.Series(["2 years", "6 months", "3 weeks", "1 day"])


def test_check_is_dog():
    animal_types = pl.Series(["Dog", "Cat", "dog", "Bird"])
    result = _check_is_dog(animal_types)
    assert result.to_list() == [1, 0, 1, 0]  # case-insensitive


def test_check_has_name():
    names = pl.Series(["Biscuit", "Rex", "unknown", "Unknown"])
    result = _check_has_name(names)
    assert result.to_list() == [True, True, False, False]


def test_get_sex_male(sex_series):
    result = _get_sex(sex_series)
    assert result[0] == "male"   # "Neutered Male" → "male"


def test_get_sex_female(sex_series):
    result = _get_sex(sex_series)
    assert result[1] == "female"  # "Spayed Female" → "female"


def test_get_sex_unknown(sex_series):
    result = _get_sex(sex_series)
    assert result[3] == "unknown"  # "Unknown" → "unknown"


def test_get_neutered_fixed(sex_series):
    result = _get_neutered(sex_series)
    assert result[0] == "fixed"   # "Neutered Male" → "fixed"
    assert result[1] == "fixed"   # "Spayed Female" → "fixed"


def test_get_neutered_intact(sex_series):
    result = _get_neutered(sex_series)
    assert result[2] == "intact"  # "Intact Male" → "intact"


def test_compute_age_days_years(age_series):
    result = _compute_age_days(age_series)
    assert result[0] == 730  # 2 years = 730 days


def test_compute_age_days_months(age_series):
    result = _compute_age_days(age_series)
    assert result[1] == 180  # 6 months = 180 days


def test_compute_age_days_weeks(age_series):
    result = _compute_age_days(age_series)
    assert result[2] == 21  # 3 weeks = 21 days


def test_add_features_output_columns():
    """add_features() must produce all expected columns."""
    df = pl.DataFrame({
        "animal_type":      ["Dog"],
        "name":             ["Biscuit"],
        "sex_upon_outcome": ["Neutered Male"],
        "age_upon_outcome": ["2 years"],
        "outcome_type":     ["Adoption"],
    })
    result = add_features(df)
    for col in ["is_dog", "has_name", "sex", "neutered", "age_days"]:
        assert col in result.columns, f"Missing column: {col}"


def test_add_features_no_mutation():
    """add_features() must not modify the original DataFrame."""
    df = pl.DataFrame({
        "animal_type":      ["Dog"],
        "name":             ["Biscuit"],
        "sex_upon_outcome": ["Neutered Male"],
        "age_upon_outcome": ["2 years"],
        "outcome_type":     ["Adoption"],
    })
    original_cols = df.columns.copy()
    _ = add_features(df)
    assert df.columns == original_cols  # Polars is immutable — this should always pass!


In [ ]:
# Run your tests directly in the notebook
# You should see all tests pass (or get clear error messages for the TODOs)
!python -m pytest {__vsc_ipynb_file__ if '__vsc_ipynb_file__' in dir() else 'day3_morning_review.ipynb'} -v 2>/dev/null || pytest.main(["-v", "--no-header", "-x"])

> ### 🟦 Trainer note — discussion question
> 
> After the exercise, ask: *"Did anyone test that `add_features` doesn't mutate the input? Did the test pass without doing anything special? Why?"*
> 
> Expected: Yes — Polars DataFrames are immutable, so this is guaranteed by the library. The test still has value as documentation of intent, but it can never fail. Contrast with the pandas version where they had to use `.copy()` explicitly.

---

## Part 4 — Logging (20 min)

> ### 🟦 Trainer note
> **Timing:** ~20 minutes.
> 
> **Recap prompt (2 min):** *"What are the five log levels in order? When would you use DEBUG vs INFO?"*
> DEBUG = internal detail useful when debugging; INFO = normal operational messages; WARNING/ERROR/CRITICAL = problems.
> 
> **Key point to reinforce:** Use `logger = logging.getLogger(__name__)` — never configure the root logger in a library (only in applications/scripts).
> 
> **What to watch for:**
> - Participants sometimes use `print()` inside the function — remind them that's fine for a notebook but the goal is to see how `logging` differs
> - The `setup_logger` function is provided and complete — they don't need to understand every line, just call it and use `logger`
> - The decorator exercise (4b) is the interesting part — it's a pattern they'll use in production

### <mark>Exercise 4a: Add logging to `add_features`</mark>

Modify your `add_features()` function to log:
- At `INFO` level: how many rows are being processed
- At `DEBUG` level: the column names of the input DataFrame
- At `INFO` level: a confirmation message when done

In [ ]:
# ── Logger setup — provided, just run this ───────────────────────────────────
def setup_logger(level: int = logging.INFO) -> None:
    """Set up the animal_shelter logger."""
    logger = logging.getLogger("animal_shelter")
    if logger.handlers:
        logger.handlers.clear()
    logger.setLevel(level)
    handler = logging.StreamHandler()
    handler.setLevel(level)
    handler.setFormatter(logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    ))
    logger.addHandler(handler)

setup_logger(level=logging.DEBUG)

# Get the logger — use this in your functions
logger = logging.getLogger("animal_shelter")

In [ ]:
def add_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add all engineered features to the DataFrame."""
    logger.info("Processing %d rows", df.height)
    logger.debug("Input columns: %s", df.columns)

    result = (
        df
        .with_columns(_check_is_dog(df["animal_type"]).alias("is_dog"))
        .with_columns(_check_has_name(df["name"]).alias("has_name"))
        .with_columns(_get_sex(df["sex_upon_outcome"]).alias("sex"))
        .with_columns(_get_neutered(df["sex_upon_outcome"]).alias("neutered"))
        .with_columns(_compute_age_days(df["age_upon_outcome"]).alias("age_days"))
    )

    logger.info("Feature engineering complete — %d columns added", result.width - df.width)
    return result


# Test it — you should see log output
add_features(sample)


### <mark>Exercise 4b: Write a `log_step` decorator</mark>

Write a decorator called `log_step` that, when applied to a pipeline function:
1. Logs the function name and input row count at `INFO` level
2. Runs the function
3. Logs the output row count (and how many rows were dropped, if any)

This is the same pattern from the `Modern_Pipelines_Polars.ipynb` notebook.

In [ ]:
def log_step(func):
    """Decorator that logs input/output row counts for a pipeline step."""
    @wraps(func)
    def wrapper(df: pl.DataFrame, *args, **kwargs) -> pl.DataFrame:
        logger.info("%s: starting with %d rows", func.__name__, df.height)
        result = func(df, *args, **kwargs)
        dropped = df.height - result.height
        logger.info(
            "%s: done — %d rows out%s",
            func.__name__,
            result.height,
            f" ({dropped} dropped)" if dropped else "",
        )
        return result
    return wrapper


# Apply your decorator to a pipeline step
@log_step
def remove_unknown_outcomes(df: pl.DataFrame) -> pl.DataFrame:
    """Drop rows where outcome_type is unknown."""
    return df.filter(pl.col("outcome_type") != "Unknown")


@log_step
def remove_missing_ages(df: pl.DataFrame) -> pl.DataFrame:
    """Drop rows where age is missing or 'Unknown'."""
    return df.filter(pl.col("age_upon_outcome") != "Unknown")


# Test the decorator
test_df = pl.DataFrame({
    "animal_type":      ["Dog", "Cat",     "Dog",    "Cat"],
    "name":             ["Rex", "unknown", "Biscuit", "Luna"],
    "sex_upon_outcome": ["Neutered Male", "Intact Female", "Spayed Female", "Unknown"],
    "age_upon_outcome": ["2 years", "Unknown", "6 months", "1 year"],
    "outcome_type":     ["Adoption", "Transfer", "Unknown", "Return to Owner"],
})

cleaned = (
    test_df
    .pipe(remove_unknown_outcomes)
    .pipe(remove_missing_ages)
)
# You should see log output showing rows going from 4 → 3 → 2
cleaned


---

## Part 5 — Context managers (20 min)

> ### 🟦 Trainer note
> **Timing:** ~20 minutes.
> 
> **Recap prompt (2 min):** *"What are the three things a context manager does? What are the two ways to write one?"*
> Three things: set up, run your code, tear down. Two ways: class with `__enter__`/`__exit__`, or generator with `@contextmanager`.
> 
> **What to watch for:**
> - Exercise 5a: `@contextmanager` generator approach is simpler — nudge participants toward it
> - Exercise 5b: the timer should work even if an exception is raised inside the block — use `try/finally`
> - The `DataFrameCheckpoint` exercise (5c) is the most original — it's a context manager that holds ML state. Make sure participants understand what it's solving before they code it.

### <mark>Exercise 5a: A `timer` context manager</mark>

Write a context manager called `timer` that measures how long a block of code takes and logs the result.

Use the `@contextmanager` generator approach from the OOP notebooks.

In [ ]:
@contextmanager
def timer(description: str = "block"):
    """Context manager that logs the execution time of the enclosed block.

    Usage:
        with timer("preprocessing"):
            result = add_features(df)
    """
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        logger.info("%s took %.4fs", description, elapsed)


# Test it
with timer("feature engineering"):
    result = add_features(sample)

# You should see a log line like:
# INFO - feature engineering took 0.0012s


### <mark>Exercise 5b: A `DataFrameCheckpoint` context manager</mark>

Write a context manager called `DataFrameCheckpoint` that:
1. On entry: saves a snapshot of a DataFrame to a temporary CSV file and logs where it saved it
2. Yields the path to that file (so the `as` variable gives you the path)
3. On exit: deletes the temporary file and logs that it was cleaned up

This is useful in ML pipelines when you want to inspect intermediate data without keeping it around permanently.

In [ ]:
import tempfile
import os

@contextmanager
def dataframe_checkpoint(df: pl.DataFrame, name: str = "checkpoint"):
    """Save a DataFrame snapshot to a temp file, yield the path, then clean up.

    Usage:
        with dataframe_checkpoint(my_df, "after_cleaning") as path:
            print(f"Inspect at: {path}")
            # file exists here
        # file is deleted here
    """
    tmp = tempfile.NamedTemporaryFile(suffix=".csv", delete=False, prefix=f"{name}_")
    path = tmp.name
    tmp.close()
    try:
        df.write_csv(path)
        logger.info("Checkpoint '%s' saved to %s (%d rows)", name, path, df.height)
        yield path
    finally:
        if os.path.exists(path):
            os.remove(path)
            logger.info("Checkpoint '%s' cleaned up", name)


# Test it
with dataframe_checkpoint(sample, "sample_data") as path:
    print(f"File exists during context: {Path(path).exists()}")
    reloaded = pl.read_csv(path)
    print(f"Rows in checkpoint: {reloaded.height}")

print(f"File exists after context: {Path(path).exists()}")


---

## Part 6 — Classes, inheritance & dunders (25 min)

> ### 🟦 Trainer note
> **Timing:** ~25 minutes.
> 
> **Recap prompt (2 min):** *"What's the difference between `__str__` and `__repr__`? When would you use a class with `__enter__`/`__exit__` rather than a `@contextmanager`?"*
> `__str__` = human-readable (for `print`); `__repr__` = unambiguous, for debugging. Class-based context managers are better when you need to store state across `__enter__` and `__exit__`.
> 
> **What to watch for:**
> - `DataPipeline.__init__` should store the steps and an empty results log — not run anything yet
> - `__call__` is a new dunder — introduce it briefly: "it lets an object be called like a function"
> - `__len__` should return the number of *steps*, not the number of rows processed
> - The child class exercise (6b) should override `run()` but reuse `__len__`, `__str__` etc.
> 
> **Nice talking point:** This `DataPipeline` class is a simplified version of sklearn's `Pipeline`. The real sklearn `Pipeline` uses the same ideas — `__len__`, `__getitem__`, and a `fit/transform` interface.

### <mark>Exercise 6a: Build a `DataPipeline` class</mark>

Build a class that wraps a sequence of pipeline steps (functions that take and return a `pl.DataFrame`) and runs them in order.

It should implement:
- `__init__(self, steps)` — store the list of step functions
- `__len__` — return the number of steps
- `__repr__` — return a useful string like `DataPipeline(steps=['remove_nulls', 'add_features'])`
- `run(self, df)` — apply each step in order, log each step name, return the final DataFrame

In [ ]:
class DataPipeline:
    """A simple sequential pipeline of DataFrame transformation steps."""

    def __init__(self, steps: list):
        self._steps = steps

    def __len__(self) -> int:
        return len(self._steps)

    def __repr__(self) -> str:
        names = [s.__name__ for s in self._steps]
        return f"DataPipeline(steps={names})"

    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps in order and return the result."""
        for step in self._steps:
            logger.info("Running step: %s", step.__name__)
            df = step(df)
        return df


# Test it
pipeline = DataPipeline(steps=[
    remove_unknown_outcomes,
    remove_missing_ages,
    add_features,
])

print(pipeline)          # should show step names
print(len(pipeline))     # should be 3

output = pipeline.run(test_df)
output


### <mark>Exercise 6b: A child class — `TimedPipeline`</mark>

Create a child class `TimedPipeline` that inherits from `DataPipeline` and overrides `run()` to also:
- Record how long each step takes
- After all steps complete, print a summary table showing step name and time taken

Use your `timer` context manager from Part 5 if you finished it, otherwise use `time.perf_counter()` directly.

In [ ]:
class TimedPipeline(DataPipeline):
    """A DataPipeline that records and reports the time taken by each step."""

    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps and print a timing summary."""
        timings = []
        for step in self._steps:
            logger.info("Running step: %s", step.__name__)
            start = time.perf_counter()
            df = step(df)
            elapsed = time.perf_counter() - start
            timings.append({"step": step.__name__, "seconds": round(elapsed, 6)})

        summary = pl.DataFrame(timings)
        print("\n── Timing summary ──────────────────")
        print(summary)
        print(f"Total: {summary['seconds'].sum():.6f}s")
        return df


# Test it
timed = TimedPipeline(steps=[
    remove_unknown_outcomes,
    remove_missing_ages,
    add_features,
])

output = timed.run(test_df)
# Expected: same output as before, plus a timing table printed to the console


---

## Part 7 — Putting it all together (20 min)

> ### 🟦 Trainer note
> **Timing:** ~20 minutes. This is the capstone exercise — meant to be slightly stretch. Some participants may not finish, and that's fine. The goal is to see how all the patterns fit together.
> 
> **What to watch for:**
> - The `__iter__` / `__next__` dunder methods turn the pipeline into a generator — this is a nice callback to the Iterators notebook
> - The `with` usage in Exercise 7b calls `__enter__` / `__exit__` — again, a callback
> - If participants are stuck, suggest they look back at the OOP notebooks for dunder method signatures
> 
> **Discussion to close the session (~5 min):**
> Ask participants: *"Which of these patterns do you think you'll use most in your own work? Which surprised you?"*
> Then briefly preview the afternoon: hackathon / free build time.

You now have all the ingredients. This final exercise brings them together into a mini ML-ready pipeline.

### <mark>Exercise 7: Extend `DataPipeline` to support iteration and use-as-context-manager</mark>

Add two more dunder methods to your `DataPipeline`:

**Part A** — Make it iterable with `__iter__` and `__next__` so you can loop over its steps:
```python
for step in pipeline:
    print(step.__name__)
```

**Part B** — Make it a context manager with `__enter__` and `__exit__` so it can be used like:
```python
with DataPipeline(steps=[...]) as pipeline:
    result = pipeline.run(df)
# On exit: log how many steps ran and that the pipeline is done
```

In [ ]:
class DataPipeline:
    """A sequential DataFrame pipeline with logging, iteration, and context manager support."""

    def __init__(self, steps: list):
        self._steps = steps
        self._current = 0  # needed for __next__

    def __len__(self) -> int:
        return len(self._steps)

    def __repr__(self) -> str:
        names = [s.__name__ for s in self._steps]
        return f"DataPipeline(steps={names})"

    def __iter__(self):
        self._current = 0
        return self

    def __next__(self):
        if self._current >= len(self._steps):
            raise StopIteration
        step = self._steps[self._current]
        self._current += 1
        return step

    def __enter__(self):
        logger.info("Pipeline starting (%d steps)", len(self._steps))
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            logger.info("Pipeline finished successfully")
        else:
            logger.error("Pipeline failed with %s: %s", exc_type.__name__, exc_val)
        return False  # do not suppress exceptions

    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps in order."""
        for step in self:
            logger.info("Running step: %s", step.__name__)
            df = step(df)
        return df


# Part A — iteration
pipeline = DataPipeline(steps=[remove_unknown_outcomes, remove_missing_ages, add_features])

print("Steps in the pipeline:")
for step in pipeline:
    print(" -", step.__name__)

print()

# Part B — context manager
with DataPipeline(steps=[remove_unknown_outcomes, remove_missing_ages, add_features]) as p:
    result = p.run(test_df)

result


---

## ✅ Session complete

Here's what you practised today:

| Pattern | Where you used it |
|---|---|
| Single-responsibility functions | `_check_is_dog`, `_get_sex`, etc. in Part 2 |
| Unit tests + fixtures | Part 3 |
| `logging` + decorators | `log_step` in Part 4 |
| `@contextmanager` with `try/finally` | `timer`, `dataframe_checkpoint` in Part 5 |
| Class with `__len__`, `__repr__` | `DataPipeline` in Part 6 |
| Inheritance | `TimedPipeline(DataPipeline)` in Part 6 |
| `__iter__` / `__next__` | Part 7a |
| `__enter__` / `__exit__` | Part 7b |

> ### 🟦 Trainer note — closing
> 
> Ask the group: *"Which pattern felt most natural? Which one would you not have reached for before this course?"*
> 
> Afternoon session suggestion: open hackathon where participants extend this pipeline to:
> - Load real data from `../data/train.csv`
> - Add a model training step to the pipeline
> - Expose it as an API or CLI using the `07_hackathon_polars` scaffold